# Pyroxene TEM Quantification Workflow

This notebook demonstrates the `StoichiometryCore` workflow for TEM-EDS quantification of a pyroxene (diopside-hedenbergite solid solution). It walks through four progressive analysis steps:

1. **K-Factored Quantification** — raw counts to weight/atomic % using instrument-specific k-factors
2. **Thickness Correction** — applies thin-film absorption correction (ρ·τ) and low takeoff angle
3. **Instrument Correction** — adds arbitrary absorption from the detector window
4. **Pyroxene Phase Analysis** — calculates cation ratios, site occupancies, and charge balance

## 1. Setup

In [1]:
import sys
import os
import warnings
os.chdir(os.path.abspath('..'))
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import StoichiometryCore as core
import PhysicsBasics as pb

# Suppress harmless RuntimeWarnings from AbsorptionCorrection.py
# when processing elements with zero abundance in the sample.
warnings.filterwarnings('ignore', category=RuntimeWarning, module='AbsorptionCorrection')

pd.set_option('display.precision', 4)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 10)

print(f'numpy {np.__version__} | pandas {pd.__version__}')
print(f'PhysicsBasics: H={pb.H} ... Uuo={pb.Uuo} (MAXELEMENT={pb.MAXELEMENT})')


numpy 2.3.5 | pandas 2.3.3
PhysicsBasics: H=1 ... Uuo=118 (MAXELEMENT=118)


## 2. TEM Count Data

Create a 118-element numpy vector with non-zero TEM-EDS counts for O, Mg, Si, Fe, Ca, Mn, Cr — representative of a diopside-hedenbergite (Ca(Mg,Fe)Si₂O₆) pyroxene grain analyzed on a JEOL 2010F TEM at 80 keV.

In [2]:
tem_counts = np.zeros(pb.MAXELEMENT)
tem_counts[pb.O-1]  = 50000
tem_counts[pb.Mg-1] = 35000
tem_counts[pb.Si-1] = 40000
tem_counts[pb.Fe-1] = 20000
tem_counts[pb.Ca-1] = 28000
tem_counts[pb.Mn-1] = 1500
tem_counts[pb.Cr-1] = 1200

nonzero = [(pb.ElementalSymbols[i], tem_counts[i]) for i in range(pb.MAXELEMENT) if tem_counts[i] > 0]
df_counts = pd.DataFrame(nonzero, columns=['Element', 'Counts'])
df_counts

,Element,Counts
0,N,50000.0
1,Na,35000.0
2,Al,40000.0
3,K,28000.0
4,V,1200.0
5,Cr,1500.0
6,Mn,20000.0


## 3. Stoichiometry

Load the silicate stoichiometry file, which provides cation charge states for the oxygen-by-stoichiometry calculation.

In [3]:
silicate_stoich = core.load_stoichiometry('Silicates')
stoich_df = pd.DataFrame({
    'Element': pb.ElementalSymbols[1:],
    'Charge': silicate_stoich
})
stoich_df[stoich_df['Charge'] != 0]

,Element,Charge
0,H,1.0
2,Li,1.0
3,Be,2.0
4,B,3.0
5,C,4.0
6,N,-3.0
7,O,-2.0
8,F,-1.0
10,Na,1.0
11,Mg,2.0


## 4. Step 1 — K-Factored Quantification

Convert raw TEM counts to weight % and atomic % using the Titan 80 keV k-factors. Oxygen is determined by stoichiometry (oxygen-by-stoichiometry method).

In [4]:
analysis_input = core.AnalysisInput(
    values=dict(core.vector_to_element_dict(tem_counts)),
    input_type='Counts',
    stoichiometry=silicate_stoich
)

options_step1 = core.AnalysisOptions(kfactors='Titan 80 keV')
result1 = core.run_analysis(analysis_input, options=options_step1)

print(result1.report_text)

quant_df1 = pd.DataFrame(result1.tables[1].rows)
quant_df1

[1. 0. 1. 2. 3. 4. 0. 0. 0. 0. 1. 2. 3. 4. 5. 0. 0. 0. 1. 2. 3. 4. 3. 3.
 2. 2. 2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 1. 2. 3. 4. 0. 0. 0. 2. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Input data:
Element           Counts
O              50000.000
Mg             35000.000
Si             40000.000
Ca             28000.000
Cr              1200.000
Mn              1500.000
Fe             20000.000
Total:        175700.000

k-factors used: Titan 80 keV
Oxygen determined by stoichiometry

Quantification results:
Element       At%      Wt%  Ox Wt %  Valence k-factor
O          57.185   38.872    0.000    0.000    1.130
Mg         13.198   13.629   22.600    2.000    0.919
Si         14.204   16.948   36.258    4.000    1.000
Ca          8.877   15.115   21.148    2.000    1.274
Cr          0.334    0.737    1.078    3.00

,element,at_pct,wt_pct,oxide_wt_pct,valence,k_factor
0,O,57.1853,38.8715,0.0000,0.0,1.130
1,Mg,13.1982,13.6286,22.6001,2.0,0.919
2,Si,14.2038,16.9484,36.2583,4.0,1.000
3,Ca,8.8766,15.1146,21.1484,2.0,1.274
4,Cr,0.3337,0.7373,1.0775,3.0,1.450
5,Mn,0.4234,0.9883,1.2761,2.0,1.555
6,Fe,5.7790,13.7113,17.6395,2.0,1.618


## 5. Step 2 — Thickness Correction

Apply a thin-film absorption correction. For a 100 nm thick specimen with 3.2 g/cm³ density: ρ·τ = 0.032 g/cm² = 32 nm·g/cm². The negative sign indicates the absorption product. Takeoff angle is 18° (typical TEM geometry).

In [5]:
options_step2 = core.AnalysisOptions(
    kfactors='Titan 80 keV',
    absorption_correction=-0.032,
    takeoff=18.0
)
result2 = core.run_analysis(analysis_input, options=options_step2)

print(result2.report_text)

quant_df2 = pd.DataFrame(result2.tables[1].rows)

# Comparison vs Step 1
comparison2 = quant_df2[['element']].copy()
comparison2['Step1 At%'] = quant_df1.set_index('element').loc[comparison2['element'], 'at_pct'].values
comparison2['Step2 At%'] = quant_df2['at_pct']
comparison2['Δ At%'] = (comparison2['Step2 At%'] - comparison2['Step1 At%']).round(4)
comparison2['Step1 Wt%'] = quant_df1.set_index('element').loc[comparison2['element'], 'wt_pct'].values
comparison2['Step2 Wt%'] = quant_df2['wt_pct']
comparison2['Δ Wt%'] = (comparison2['Step2 Wt%'] - comparison2['Step1 Wt%']).round(4)
comparison2

Loading CXRO Scattering Files/o.nff
Loading CXRO Scattering Files/mg.nff
Loading CXRO Scattering Files/si.nff
Loading CXRO Scattering Files/ca.nff
Loading CXRO Scattering Files/cr.nff
Loading CXRO Scattering Files/mn.nff
Loading CXRO Scattering Files/fe.nff
[1. 0. 1. 2. 3. 4. 0. 0. 0. 0. 1. 2. 3. 4. 5. 0. 0. 0. 1. 2. 3. 4. 3. 3.
 2. 2. 2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 1. 2. 3. 4. 0. 0. 0. 2. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Input data:
Element           Counts
O              50000.000
Mg             35000.000
Si             40000.000
Ca             28000.000
Cr              1200.000
Mn              1500.000
Fe             20000.000
Total:        175700.000

k-factors used: Titan 80 keV
Absorption Correction: -32.000000 nm*g/cm3
Takeoff angle: 18.00 degrees
Oxygen determined by stoichiometry

Quantif

,element,Step1 At%,Step2 At%,Δ At%,Step1 Wt%,Step2 Wt%,Δ Wt%
0,O,57.1853,57.1817,-0.0036,38.8715,38.8310,-0.0405
1,Mg,13.1982,13.1093,-0.0889,13.6286,13.5236,-0.1050
2,Si,14.2038,14.1955,-0.0083,16.9484,16.9219,-0.0265
3,Ca,8.8766,8.9311,0.0545,15.1146,15.1925,0.0780
4,Cr,0.3337,0.3360,0.0023,0.7373,0.7415,0.0043
5,Mn,0.4234,0.4264,0.0030,0.9883,0.9942,0.0059
6,Fe,5.7790,5.8200,0.0411,13.7113,13.7952,0.0839


## 6. Step 3 — Instrument Correction

Add the arbitrary absorption correction for the Titan detector window, on top of the thickness correction. This accounts for low-energy X-ray absorption in the detector filter/window before the X-rays reach the detector.

In [6]:
options_step3 = core.AnalysisOptions(
    kfactors='Titan 80 keV',
    absorption_correction=-0.032,
    arbitrary_absorption='Titan Detector Window',
    takeoff=18.0
)
result3 = core.run_analysis(analysis_input, options=options_step3)

print(result3.report_text)

quant_df3 = pd.DataFrame(result3.tables[1].rows)

# Full comparison across all three steps
comparison3 = quant_df3[['element']].copy()
comparison3['Step1 At%'] = quant_df1.set_index('element').loc[comparison3['element'], 'at_pct'].values
comparison3['Step2 At%'] = quant_df2.set_index('element').loc[comparison3['element'], 'at_pct'].values
comparison3['Step3 At%'] = quant_df3['at_pct']
comparison3['Step1 Wt%'] = quant_df1.set_index('element').loc[comparison3['element'], 'wt_pct'].values
comparison3['Step2 Wt%'] = quant_df2.set_index('element').loc[comparison3['element'], 'wt_pct'].values
comparison3['Step3 Wt%'] = quant_df3['wt_pct']
comparison3

Loading CXRO Scattering Files/be.nff
Loading CXRO Scattering Files/b.nff
Loading CXRO Scattering Files/c.nff
Loading CXRO Scattering Files/n.nff
Loading CXRO Scattering Files/al.nff
Titan Detector Window
[1. 0. 1. 2. 3. 4. 0. 0. 0. 0. 1. 2. 3. 4. 5. 0. 0. 0. 1. 2. 3. 4. 3. 3.
 2. 2. 2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 1. 2. 3. 4. 0. 0. 0. 2. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Input data:
Element           Counts
O              50000.000
Mg             35000.000
Si             40000.000
Ca             28000.000
Cr              1200.000
Mn              1500.000
Fe             20000.000
Total:        175700.000

Arbitrary Absorption Correction used: Titan Detector Window
k-factors used: Titan 80 keV
Absorption Correction: -32.000000 nm*g/cm3
Takeoff angle: 18.00 degrees
Oxygen determined by stoichiometry

Q

,element,Step1 At%,Step2 At%,Step3 At%,Step1 Wt%,Step2 Wt%,Step3 Wt%
0,O,57.1853,57.1817,57.1685,38.8715,38.8310,38.8887
1,Mg,13.1982,13.1093,13.3154,13.6286,13.5236,13.7598
2,Si,14.2038,14.1955,14.1712,16.9484,16.9219,16.9220
3,Ca,8.8766,8.9311,8.8523,15.1146,15.1925,15.0842
4,Cr,0.3337,0.3360,0.3317,0.7373,0.7415,0.7332
5,Mn,0.4234,0.4264,0.4207,0.9883,0.9942,0.9826
6,Fe,5.7790,5.8200,5.7403,13.7113,13.7952,13.6295


## 7. Step 4 — Pyroxene Phase Analysis

Run the pyroxene ideal phase analysis plugin, which calculates:
- Cation ratios (Mg/(Mg+Fe), element/(M1+M2))
- Cations per 6 oxygens
- Site occupancies for T (tetrahedral), M1 (octahedral), and M2 (octahedral) sites
- Charge balance assessment

Based on the Morimoto et al. (1988) pyroxene nomenclature system.

In [7]:
options_step4 = core.AnalysisOptions(
    kfactors='Titan 80 keV',
    absorption_correction=-0.032,
    arbitrary_absorption='Titan Detector Window',
    takeoff=18.0,
    phase_analysis='Pyroxene ideal'
)
result4 = core.run_analysis(analysis_input, options=options_step4)

print(result4.report_text)

Titan Detector Window
[1. 0. 1. 2. 3. 4. 0. 0. 0. 0. 1. 2. 3. 4. 5. 0. 0. 0. 1. 2. 3. 4. 3. 3.
 2. 2. 2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 1. 2. 3. 4. 0. 0. 0. 2. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Input data:
Element           Counts
O              50000.000
Mg             35000.000
Si             40000.000
Ca             28000.000
Cr              1200.000
Mn              1500.000
Fe             20000.000
Total:        175700.000

Arbitrary Absorption Correction used: Titan Detector Window
k-factors used: Titan 80 keV
Absorption Correction: -32.000000 nm*g/cm3
Takeoff angle: 18.00 degrees
Oxygen determined by stoichiometry

Quantification results:
Element       At%      Wt%  Ox Wt %  Valence k-factor
O          57.169   38.889    0.000    0.000    1.130
Mg         13.315   13.760   22.817    2.000    0.9

In [8]:
# Parse phase analysis tables as DataFrames
phase_tables = [t for t in result4.tables if t.name.startswith('phase_analysis_')]

for table in phase_tables:
    print(f'\n### {table.title}')
    df_phase = pd.DataFrame(table.rows, columns=table.columns)
    display(df_phase)


### Pyroxene ideal Numeric Results


,metric,value,note
0,Mg/(Mg+Fe),0.699,
1,Fe/(M1+M2),0.200,
2,Cr/(M1+M2),0.012,
3,Mg/(M1+M2),0.465,
4,Mn/(M1+M2),0.015,
5,Ca/(M1+M2),0.309,
6,Assumed M1+M2,1.000,"which is everything except O, Si and Al"
7,Ideal sum cations,4.000,", Ideal sum charge = 12"



### Pyroxene ideal - Cations per 6 oxygens


,label,value
0,Si,1.487
1,Fe,0.602
2,Cr,0.035
3,Mg,1.397
4,Mn,0.044
5,Ca,0.929
6,Total Cats,4.495



### Pyroxene ideal - Assuming exactly 4 cations and compute to fill sites and balance charge


,label,value
0,Si4+,1.323
1,Fe3+,0.536
2,Total occ,1.860
3,Charge,6.902



### Pyroxene ideal - Site M1 (octahedral)


,label,value
0,Cr3+,0.031
1,Mg2+,0.969
2,Total occ,1.000
3,Charge,2.031



### Pyroxene ideal - Site M2 (octahedral)


,label,value
0,Mg2+,0.274
1,Mn2+,0.039
2,Ca2+,0.827
3,Total occ,1.140
4,Charge,2.281


## 8. Summary

Each step adds a layer of physical correction to the quantification:

| Step | Corrections Applied | Purpose |
|------|--------------------|---------|
| 1 | K-factors (Titan 80 keV) | Convert raw counts to elemental abundances |
| 2 | + Thickness (-0.032 g/cm²), takeoff 18° | Account for thin-film X-ray absorption in the specimen |
| 3 | + Arbitrary absorption (Titan Detector Window) | Account for X-ray absorption in the detector window/filter |
| 4 | + Pyroxene ideal phase analysis | Calculate mineral-specific cation ratios and site occupancies |

**Note:** All corrections can be combined in a single `run_analysis` call (as shown in Step 4). The stepwise approach above demonstrates the individual contribution of each correction layer.